In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeRegressor

In [5]:
df = pd.read_csv('card_cust.csv')
df.head(2)

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,10001,40.900749,0.818182,95.4,0.0,95.4,0.000000,0.166667,0.0,0.083333,0.00,0.0,2.0,1000.0,201.802084,139.509787,0.000000,12.0
1,10002,3202.467416,0.909091,0.0,0.0,0.0,6442.945483,0.000000,0.0,0.000000,0.25,4.0,0.0,7000.0,4103.032597,1072.340217,0.222222,12.0


## 전처리
### 결측치
- 분석 수행 전 '기한 내 최소 지불 금액(MINIMUM_PAYMENTS)'의 결측 값(Null)을 각 컬럼의 평균값으로
대체하시오.


In [8]:
# 결측치 여부를 확인
df.isna().sum()
#dataframe의 기준을 col 이다. 그래서 isna()를 할 때 col 의 결측치를 계산하고 결과를 붙여보여주는 것
#sum() 역시 col의 합을 보여주는 것이고 그게 옆으로 기니 아래로 붙여서 보여준 것

,0
CUST_ID,0
BALANCE,0
BALANCE_FREQUENCY,0
PURCHASES,0
ONEOFF_PURCHASES,0
INSTALLMENTS_PURCHASES,0
CASH_ADVANCE,0
PURCHASES_FREQUENCY,0
ONEOFF_PURCHASES_FREQUENCY,0
PURCHASES_INSTALLMENTS_FREQUENCY,0


In [9]:
df['MINIMUM_PAYMENTS'] = df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].mean())

In [11]:
df.isna().sum().sum()

np.int64(0)

In [12]:
df_base=df.copy()

# Q01.
(base를 사용하여)***연간 평균 잔고액과 신용카드 서비스 이용기간 간의 관계를
파악***하여, 추후 고객의 신용카드 한도 조정에 근거 자료로 활용하고자 한다.
연간 평균 잔고액(BALANCE)이 많을수록, 그리고 신용카드 서비스 이용기간
(TENURE)이 길수록 신용카드 한도(CREDIT_LIMIT) 역시 높을 것으로 예상해볼 수
있다. ***신용 카드 서비스 이용기간(TENURE) 별로 연간 평균 잔고액 (BALANCE)과
신용카드 한도(CREDIT_LIMIT) 간 피어슨(Pearson)
상관 분석을 실시하고, 이 중 가장 큰 상관계수를 구하시오.***

※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오.(정답 예시: 0.12)

In [14]:
df_q1 = df_base[["TENURE","BALANCE","CREDIT_LIMIT"]].copy()
df_q1.head(2)

,TENURE,BALANCE,CREDIT_LIMIT
0,12.0,40.900749,1000.0
1,12.0,3202.467416,7000.0


groupby('A')[['대상1','대상2']] 까지는 dataframe 객체로 두지 않는다.

## 데이터

|Index | TENURE | BALANCE | CREDIT\_LIMIT |
|--| ------ | ------- | ------------- |
|0| 12     | 100     | 1000          |
|1| 12     | 200     | 1500          |
|2| 6      | 150     | 800           |
|3| 6      | 300     | 1200          |
|4| 24     | 250     | 2000          |


## TENURE를 기준으로 groupby한다면


#### 그룹: TENURE = 12
| index | BALANCE | CREDIT\_LIMIT |
| ----- | ------- | ------------- |
| 2     | 150     | 800           |
| 3     | 300     | 1200          |

#### 그룹: TENURE = 6
| index | BALANCE | CREDIT\_LIMIT |
| ----- | ------- | ------------- |
| 0     | 100     | 1000          |
| 1     | 200     | 1500          |

#### 그룹: TENURE = 24
| index | BALANCE | CREDIT\_LIMIT |
| ----- | ------- | ------------- |
| 4     | 250     | 2000          |

위와 같이 그루핑만 된다.
그래서 col의 연산(mean, sum) 또는 col 간 연산(corr()) 등을 수행하여 dataframe으로 표현하는 것이다.
#### groupby를 하게되면 mean(),sum(),corr() 등은 필연일 것!

## groupby 후 mean() 할 경우 dataframe 생성

| TENURE | BALANCE | CREDIT\_LIMIT |
| ------ | ------- | ------------- |
| 6      | 225.0   | 1000.0        |
| 12     | 150.0   | 1250.0        |
| 24     | 250.0   | 2000.0        |

## groupby 후 corr() 할 경우 dataframe 생성
| TENURE |               | BALANCE | CREDIT\_LIMIT |
| ------ | ------------- | ------- | ------------- |
| 6      | BALANCE       | 1.00    | 1.00          |
|       | CREDIT\_LIMIT | 1.00    | 1.00          |
| 12     | BALANCE       | 1.00    | 1.00          |
|      | CREDIT\_LIMIT | 1.00    | 1.00          |
| 24     | BALANCE       | NaN     | NaN           |
|      | CREDIT\_LIMIT | NaN     | NaN           |


In [21]:
#신용 카드 서비스 이용기간(TENURE) 별로 연간 평균 잔고액 (BALANCE)과 신용카드 한도(CREDIT_LIMIT) 간 피어슨(Pearson) 상관 분석을 실시
# A 별로 B와 C의 ... --> A의 값에 따라 B와 C를 묶는다.
df_q1_groupby=df_q1.groupby(['TENURE'])[['BALANCE','CREDIT_LIMIT']].corr()
df_q1_groupby.head()

BALANCE  CREDIT_LIMIT
TENURE                                     
6.0    BALANCE       1.000000      0.868056
       CREDIT_LIMIT  0.868056      1.000000
7.0    BALANCE       1.000000      0.948405
       CREDIT_LIMIT  0.948405      1.000000
8.0    BALANCE       1.000000      0.820696

In [23]:
df_q1_groupby=df_q1_groupby.round(2)
df_q1_groupby.head()

BALANCE  CREDIT_LIMIT
TENURE                                    
6.0    BALANCE          1.00          0.87
       CREDIT_LIMIT     0.87          1.00
7.0    BALANCE          1.00          0.95
       CREDIT_LIMIT     0.95          1.00
8.0    BALANCE          1.00          0.82

In [26]:
df_q1_groupby2 = df_q1_groupby.reset_index()
#groupby를 reset_index 하면 대분류,중분류 의 부분 중 중분류가 level_1으로 col이 만들어진다.
df_q1_groupby2.head()

,index,TENURE,level_1,BALANCE,CREDIT_LIMIT
0,0,6.0,BALANCE,1.00,0.87
1,1,6.0,CREDIT_LIMIT,0.87,1.00
2,2,7.0,BALANCE,1.00,0.95
3,3,7.0,CREDIT_LIMIT,0.95,1.00
4,4,8.0,BALANCE,1.00,0.82


In [28]:
#상관관계되는 두 변수를 각각 행,열로 하나씩 선택한다.
df_q1_groupby3=df_q1_groupby2.loc[df_q1_groupby2['level_1'] == 'BALANCE' ,'CREDIT_LIMIT']
df_q1_groupby3.head()

,CREDIT_LIMIT
0,0.87
2,0.95
4,0.82
6,0.09
8,0.29


In [29]:
df_q1_groupby3.max()

0.95

# Q02.
(base를 사용하여)전략을 수립하기 위해 고객 세분화를 수행하고자 한다.
일시불 구매 금액이 높은 고객군을 도출하기 위해 다음 단계에 따라 분석을 수행하고
질문에 답하시오.  
단계 1: '고객 ID'를 제외한 모든 변수(17개)에 대해 Z-score 표준화(Standardization) 한다.  
단계 2: 표준화된 변수들에 대해 K-means 군집 분석을 수행한다.
이 때, 군집 수는 2~5개 중 K-means Silhouette 를 통해 구한 최적의 K로 설정한다.  
단계 3: 단계 2에서 도출한 각 군집 별로 ‘일시불 구매 총액’의 평균을 계산한다.
군집 별 일시불 구매 총액(ONEOFF_PURCHASES)의 평균 중 가장 큰 값은 얼마인가?

※ 정규화를 실시하지 않은 일시불 구매 총액 데이터를 기준으로 평균을 산출하시오.
※ seed는 1234로 설정하시오.
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오.(정답 예시: 0.12)

In [30]:
df_base.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0.0,2.0,1000.0,201.802084,139.509787,0.000000,12.0
1,10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4.0,0.0,7000.0,4103.032597,1072.340217,0.222222,12.0
2,10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0.0,12.0,7500.0,622.066742,627.284787,0.000000,12.0
3,10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1.0,1.0,7500.0,0.000000,1297.116322,0.000000,12.0
4,10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0.0,1.0,1200.0,678.334763,244.791237,0.000000,12.0


#### 단계 1: '고객 ID'를 제외한 모든 변수(17개)에 대해 Z-score 표준화(Standardization) 한다.

In [31]:
df_q2 = df_base.drop(columns='CUST_ID')
df_q2.head()

,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0.0,2.0,1000.0,201.802084,139.509787,0.000000,12.0
1,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4.0,0.0,7000.0,4103.032597,1072.340217,0.222222,12.0
2,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0.0,12.0,7500.0,622.066742,627.284787,0.000000,12.0
3,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1.0,1.0,7500.0,0.000000,1297.116322,0.000000,12.0
4,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0.0,1.0,1200.0,678.334763,244.791237,0.000000,12.0


In [40]:
arr_q2_nor = StandardScaler().fit_transform(df_q2) #df -> ndarray
print(type(arr_q2_nor))
print(arr_q2_nor)
print("arr_q2_nor   " + str(arr_q2_nor.shape)) #dataframe을 2차원배열로 변경됨
print("df_q2         "+ str(df_q2.shape))

<class 'numpy.ndarray'>
[[-0.84876759 -0.41987944 -0.4419358  ... -0.44372465 -0.46554357
   0.28242902]
 [ 0.28279099  0.01213096 -0.4690169  ... -0.08615941  0.33159169
   0.28242902]
 [ 0.0296341   0.44414137 -0.24953794 ... -0.25675456 -0.46554357
   0.28242902]
 ...
 [-0.04555583  0.44414137 -0.4690169  ... -0.1126367  -0.46554357
   0.28242902]
 [ 0.18482699  0.44414137 -0.4233367  ... -0.07613978 -0.46554357
   0.28242902]
 [-0.34079285  0.44414137 -0.20275918 ...  0.00909944 -0.46554357
   0.28242902]]
arr_q2_nor   (1000, 17)
df_q2         (1000, 17)


In [42]:
df_q2_nor = pd.DataFrame(arr_q2_nor, columns=df_q2.columns)
df_q2_nor.head()

,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,-0.848768,-0.419879,-0.441936,-0.374048,-0.395301,-0.482354,-0.872701,-0.804321,-0.719620,-0.684701,-0.457918,-0.564116,-1.161669,-0.557396,-0.443725,-0.465544,0.282429
1,0.282791,0.012131,-0.469017,-0.374048,-0.470304,1.878468,-1.282558,-0.804321,-0.924403,0.513493,0.065417,-0.628057,0.150025,0.360574,-0.086159,0.331592,0.282429
2,0.029634,0.444141,-0.249538,-0.099717,-0.470304,-0.482354,1.176582,2.178206,-0.924403,-0.684701,-0.457918,-0.244413,0.259333,-0.458507,-0.256755,-0.465544,0.282429
3,-0.266887,-1.283900,-0.043497,0.157817,-0.470304,-0.406949,-1.077631,-0.555778,-0.924403,-0.285304,-0.327084,-0.596086,0.259333,-0.604881,0.000000,-0.465544,0.282429
4,-0.570738,0.444141,-0.464475,-0.368371,-0.470304,-0.482354,-1.077631,-0.555778,-0.924403,-0.684701,-0.457918,-0.596086,-1.117945,-0.445267,-0.403369,-0.465544,0.282429


In [43]:
df_q2_nor = df_q2_nor.add_suffix("_S")
df_q2_nor.head(2)

,BALANCE_S,BALANCE_FREQUENCY_S,PURCHASES_S,ONEOFF_PURCHASES_S,INSTALLMENTS_PURCHASES_S,CASH_ADVANCE_S,PURCHASES_FREQUENCY_S,ONEOFF_PURCHASES_FREQUENCY_S,PURCHASES_INSTALLMENTS_FREQUENCY_S,CASH_ADVANCE_FREQUENCY_S,CASH_ADVANCE_TRX_S,PURCHASES_TRX_S,CREDIT_LIMIT_S,PAYMENTS_S,MINIMUM_PAYMENTS_S,PRC_FULL_PAYMENT_S,TENURE_S
0,-0.848768,-0.419879,-0.441936,-0.374048,-0.395301,-0.482354,-0.872701,-0.804321,-0.719620,-0.684701,-0.457918,-0.564116,-1.161669,-0.557396,-0.443725,-0.465544,0.282429
1,0.282791,0.012131,-0.469017,-0.374048,-0.470304,1.878468,-1.282558,-0.804321,-0.924403,0.513493,0.065417,-0.628057,0.150025,0.360574,-0.086159,0.331592,0.282429


#### 단계 2: 표준화된 변수들에 대해 K-means 군집 분석을 수행한다. 이 때, 군집 수는 2~5개 중 K-means Silhouette 를 통해 구한 최적의 K로 설정한다.

※ seed는 1234로 설정하시오.


In [44]:
candi_K = [2,3,4,5]
k = candi_K[0]

# 모델 생성
model_kmeans = KMeans(n_clusters=k, random_state= 1234)

#모델 학습=fit
model_kmeans.fit(df_q2_nor)
model_kmeans.labels_

array([0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1,
       1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1,
       1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0,

In [45]:
silhouette_score(df_q2_nor, labels=model_kmeans.labels_)

np.float64(0.21544722357637516)

In [47]:
list_sil = []
for k in candi_K :
  # 모델생성, 군집개수,난수
  model_kmeans = KMeans(n_clusters=k, random_state=1234)
  # 모델 학습 fit(데이터)
  model_kmeans.fit(df_q2_nor)
  #실루엣 (데이터, 라벨)
  val_sil = silhouette_score(df_q2_nor, model_kmeans.labels_)
  list_sil += [val_sil]
print(list_sil)

[np.float64(0.21544722357637516), np.float64(0.19481269449969035), np.float64(0.20726811316629618), np.float64(0.20933988710845602)]


In [49]:
series_sil = pd.Series(list_sil, index=candi_K)
series_sil

,0
2,0.215447
3,0.194813
4,0.207268
5,0.209340


In [53]:

best_k = series_sil.idxmax()
series_sil.max(), series_sil.idxmax()

(0.21544722357637516, np.int64(2))

#### 단계 3: 단계 2에서 도출한 각 군집 별로 ‘일시불 구매 총액’의 평균을 계산한다. 군집 별 일시불 구매 총액(ONEOFF_PURCHASES)의 평균 중 가장 큰 값은 얼마인가?

In [55]:
# 모델 생성
model_kmeans_k2 = KMeans(n_clusters=best_k, random_state=1234)

#모델 학습
model_kmeans_k2.fit(df_q2_nor)

df_q2['cluster'] = model_kmeans_k2.labels_
df_q2.head()

,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE,cluster
0,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0.0,2.0,1000.0,201.802084,139.509787,0.000000,12.0,0
1,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4.0,0.0,7000.0,4103.032597,1072.340217,0.222222,12.0,0
2,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0.0,12.0,7500.0,622.066742,627.284787,0.000000,12.0,1
3,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1.0,1.0,7500.0,0.000000,1297.116322,0.000000,12.0,0
4,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0.0,1.0,1200.0,678.334763,244.791237,0.000000,12.0,0


In [62]:
df_q2_groupby=df_q2.groupby(['cluster'])[['ONEOFF_PURCHASES']].mean().round(2)
df_q2_groupby

,ONEOFF_PURCHASES
cluster,
0,272.26
1,2156.47


In [63]:
df_q2_groupby['ONEOFF_PURCHASES'].max()

2156.47

## 꿀팁
- A 별로 B,C 들의 (평균,합,상관관계) --> groupby A(B,C)
- 예시
  - 신용 카드 서비스 이용기간(TENURE) ***별로*** 연간 평균 잔고액 (BALANCE)과 신용카드 한도(CREDIT_LIMIT) 간 피어슨(Pearson) 상관 분석을 실시하고, 이 중 가장 큰 상관계수를 구하시오.
  - 단계 3: 단계 2에서 도출한 각 군집 ***별로*** ‘일시불 구매 총액’의 평균을 계산한다. 군집 별 일시불 구매 총액(ONEOFF_PURCHASES)의 평균 중 가장 큰 값은 얼마인가?


# Q03.
(base를 사용하여) 일시불 구매 총액(ONEOFF\_PURCHASES) 예측 모델을 Target Marketing에 활용하고자 한다. 다음 단계에 따라 분석을 수행하고 질문에 답하시오.

---

**단계 1**: ‘고객 ID(CUST\_ID)’가 4의 배수가 아닌 데이터를 Train Set으로, 4의 배수인 데이터를 Test Set으로 분할한다.⭐⭐

**단계 2**: Train Set으로 아래 조건에 따라 의사결정나무 회귀모델을 학습한다.

* **독립 변수**(총 16개): ‘고객 ID’, ‘일시불 구매 총액’을 제외한 모든 변수
* **종속 변수**: ‘일시불 구매 총액’

**단계 3**: 생성된 모델을 Test Set에 적용하여 ‘일시불 구매 총액’을 예측한다.

---

### 단계 3에서 얻은 예측 결과를 평가하기 위해, 아래 정의된 Measure B를 계산한 값은?

$$
B = \left( \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2 \right)^{\frac{1}{2}}
$$

* $\hat{y}_i$: 예측값
* $y_i$: 실제값

---

※ seed는 **1234**로 설정하시오.
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. (정답 예시: 0.12)

---


단계 1: ‘고객 ID(CUST_ID)’가 4의 배수가 아닌 데이터를 Train Set으로, 4의 배수인 데이터를 Test Set으로 분할한다.⭐⭐

In [65]:
df_train = df_base.loc[(df_base['CUST_ID']%4) != 0,]
df_test = df_base.loc[(df_base['CUST_ID']%4) == 0,]
len(df_base), len(df_train), len(df_test)

(1000, 752, 248)

**단계 2**: ***Train Set***으로 아래 조건에 따라 의사결정나무 회귀모델을 학습한다.

* **독립 변수**(총 16개): ‘고객 ID’, ‘일시불 구매 총액’을 제외한 모든 변수
* **종속 변수**: ‘일시불 구매 총액’

**단계 3**: 생성된 모델을 Test Set에 적용하여 ‘일시불 구매 총액’을 예측한다.  
--> train의 독립변수와 동일하게 제외할건 제외한다.

In [67]:
# 모델 생성
model_dtr = DecisionTreeRegressor(random_state=1234)

#모델 학습
model_dtr.fit(X = df_train.drop(columns=["CUST_ID", "ONEOFF_PURCHASES"]),
          y=df_train["ONEOFF_PURCHASES"])

#모델 예측
pred = model_dtr.predict(df_test.drop(columns=["CUST_ID", "ONEOFF_PURCHASES"]))

pred

array([1.500000e+03, 0.000000e+00, 1.490000e+03, 0.000000e+00,
       0.000000e+00, 3.322280e+03, 0.000000e+00, 3.515400e+02,
       0.000000e+00, 8.863700e+02, 2.020000e+02, 0.000000e+00,
       6.614900e+02, 0.000000e+00, 0.000000e+00, 0.000000e+00,
       2.870000e+02, 1.386400e+03, 5.719100e+02, 5.000000e+00,
       4.783340e+03, 1.282850e+03, 3.260200e+02, 0.000000e+00,
       0.000000e+00, 0.000000e+00, 0.000000e+00, 0.000000e+00,
       6.159900e+02, 0.000000e+00, 0.000000e+00, 0.000000e+00,
       9.695600e+02, 0.000000e+00, 2.225739e+04, 0.000000e+00,
       0.000000e+00, 1.234000e+03, 8.324900e+02, 6.099160e+03,
       1.850000e+02, 0.000000e+00, 0.000000e+00, 3.161460e+03,
       2.220390e+03, 0.000000e+00, 0.000000e+00, 3.132700e+02,
       0.000000e+00, 1.085190e+03, 3.200300e+02, 4.100080e+03,
       9.000000e+02, 0.000000e+00, 1.674000e+02, 3.322280e+03,
       1.383740e+03, 1.136700e+03, 0.000000e+00, 1.180000e+03,
       1.776400e+02, 5.786600e+02, 0.000000e+00, 0.0000

### 단계 3에서 얻은 예측 결과를 평가하기 위해, 아래 정의된 Measure B를 계산한 값은?

$$
B = \left( \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2 \right)^{\frac{1}{2}}
$$
--> RMSE 임을 알아야 쉽다.
* $\hat{y}_i$: 예측값
* $y_i$: 실제값

---

※ seed는 **1234**로 설정하시오  .
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. (정답 예시: 0.12)


In [69]:
from sklearn.metrics import mean_squared_error
y_true = df_test["ONEOFF_PURCHASES"]
y_pred = pred
RMSE = mean_squared_error(y_true = y_true, y_pred=y_pred)**0.5
round(RMSE, 2)

1039.19

In [71]:
# y_true - y_pred # E, Error
# (y_true - y_pred) ** 2 # SE, Squared Error
# ((y_true - y_pred) ** 2).mean() # MSE, Mean Squared Error
((y_true - y_pred) ** 2).mean() ** 0.5 # RMSE, Root Mean Squared Error

np.float64(1039.193967231063)

## 지도학습과 비지도학습 골격은 동일하다
### 1.모델생성
- 난수

### 2. 모델학습
- 지도학습
  - X=학습데이터, y=평가데이터
- 비지도학습(kemans only)
  - 학습데이터

### 3. 예측